# Q3: Data Wrangling

**Phase 4:** Data Wrangling & Transformation  

In [13]:
# Import libraries
import pandas as pd
import numpy as np
import os

# Load cleaned data from Q2
df = pd.read_csv('output/q2_cleaned_data.csv')
print(f"Loaded {len(df):,} cleaned records")

# Display first few rows
print("\nFirst few rows:")
print(df.head())

Loaded 195,892 cleaned records

First few rows:
                  Station Name   Measurement Timestamp  Air Temperature  \
0  63rd Street Weather Station  09/27/2018 10:00:00 AM            16.40   
1  63rd Street Weather Station  09/27/2018 11:00:00 AM            17.10   
2  63rd Street Weather Station  09/27/2018 01:00:00 PM            18.20   
3       Foster Weather Station  09/27/2018 01:00:00 PM            17.89   
4  63rd Street Weather Station  09/27/2018 03:00:00 PM            19.50   

   Wet Bulb Temperature  Humidity  Rain Intensity  Interval Rain  Total Rain  \
0                  12.2        61             0.0            0.0       260.3   
1                  11.5        51             0.0            0.0       260.3   
2                  12.4        51             0.0            0.0       260.3   
3                   NaN        39             NaN            0.0         NaN   
4                  13.0        47             0.0            0.0       260.3   

   Precipitation Typ

In [14]:
# ========================================
# STEP 2: PARSE DATETIME COLUMN
# ========================================
print("\n" + "="*60)
print("STEP 1: DATETIME PARSING")
print("="*60)

# Identify datetime column (adjust name as needed)
datetime_col = 'Measurement Timestamp'  # Update based on your actual column name

# Check current data type
print(f"\nCurrent data type of '{datetime_col}': {df[datetime_col].dtype}")

# Parse datetime column
print(f"\nParsing '{datetime_col}' to datetime...")
df[datetime_col] = pd.to_datetime(df[datetime_col])
print(f"✓ Successfully parsed to datetime64[ns]")

# Verify parsing
print(f"\nDate range after parsing:")
print(f"  Start: {df[datetime_col].min()}")
print(f"  End: {df[datetime_col].max()}")


STEP 1: DATETIME PARSING

Current data type of 'Measurement Timestamp': object

Parsing 'Measurement Timestamp' to datetime...
✓ Successfully parsed to datetime64[ns]

Date range after parsing:
  Start: 2015-04-25 09:00:00
  End: 2025-11-24 12:00:00


In [15]:

# ========================================
# STEP 3: SET DATETIME INDEX
# ========================================
print("\n" + "="*60)
print("STEP 2: SETTING DATETIME INDEX")
print("="*60)

# Set datetime column as index
print(f"\nSetting '{datetime_col}' as index...")
df = df.set_index(datetime_col)
print("✓ Datetime index set")

# Sort index chronologically (IMPORTANT for time series)
print("\nSorting index chronologically...")
df = df.sort_index()
print("✓ Index sorted")

# Verify index
print(f"\nIndex name: {df.index.name}")
print(f"Index type: {df.index.dtype}")
print(f"Index is sorted: {df.index.is_monotonic_increasing}")

# Display index range
print(f"\nDatetime range:")
print(f"  Start: {df.index.min()}")
print(f"  End: {df.index.max()}")


STEP 2: SETTING DATETIME INDEX

Setting 'Measurement Timestamp' as index...
✓ Datetime index set

Sorting index chronologically...
✓ Index sorted

Index name: Measurement Timestamp
Index type: datetime64[ns]
Index is sorted: True

Datetime range:
  Start: 2015-04-25 09:00:00
  End: 2025-11-24 12:00:00


In [16]:

# ========================================
# STEP 4: EXTRACT TEMPORAL FEATURES
# ========================================
print("\n" + "="*60)
print("STEP 3: EXTRACTING TEMPORAL FEATURES")
print("="*60)

print("\nExtracting required temporal features...")

# Required features
df['hour'] = df.index.hour
print("  ✓ Extracted: hour (0-23)")

df['day_of_week'] = df.index.dayofweek  # 0=Monday, 6=Sunday
print("  ✓ Extracted: day_of_week (0=Monday, 6=Sunday)")

df['month'] = df.index.month
print("  ✓ Extracted: month (1-12)")

# Optional but recommended features
print("\nExtracting optional temporal features...")

df['year'] = df.index.year
print("  ✓ Extracted: year")

df['day_name'] = df.index.day_name()
print("  ✓ Extracted: day_name (e.g., 'Monday')")

df['is_weekend'] = (df.index.dayofweek >= 5).astype(int)  # 1 if Sat/Sun, 0 otherwise
print("  ✓ Extracted: is_weekend (0 or 1)")

# Additional useful features
df['day_of_month'] = df.index.day
print("  ✓ Extracted: day_of_month (1-31)")

df['quarter'] = df.index.quarter
print("  ✓ Extracted: quarter (1-4)")

# Verify extraction
print("\nTemporal features summary:")
temporal_features = ['hour', 'day_of_week', 'month', 'year', 'day_name', 'is_weekend']
print(df[temporal_features].head(10))

# Check for any missing values in temporal features
missing_temporal = df[temporal_features].isnull().sum()
if missing_temporal.sum() > 0:
    print("\nWarning: Missing values in temporal features:")
    print(missing_temporal[missing_temporal > 0])
else:
    print("\n✓ No missing values in temporal features")



STEP 3: EXTRACTING TEMPORAL FEATURES

Extracting required temporal features...
  ✓ Extracted: hour (0-23)
  ✓ Extracted: day_of_week (0=Monday, 6=Sunday)
  ✓ Extracted: month (1-12)

Extracting optional temporal features...
  ✓ Extracted: year
  ✓ Extracted: day_name (e.g., 'Monday')
  ✓ Extracted: is_weekend (0 or 1)
  ✓ Extracted: day_of_month (1-31)
  ✓ Extracted: quarter (1-4)

Temporal features summary:
                       hour  day_of_week  month  year  day_name  is_weekend
Measurement Timestamp                                                      
2015-04-25 09:00:00       9            5      4  2015  Saturday           1
2015-04-30 05:00:00       5            3      4  2015  Thursday           0
2015-05-22 15:00:00      15            4      5  2015    Friday           0
2015-05-22 16:00:00      16            4      5  2015    Friday           0
2015-05-22 17:00:00      17            4      5  2015    Friday           0
2015-05-22 17:00:00      17            4      5  2015  

In [17]:

# ========================================
# STEP 5: CALCULATE DATE RANGE INFO
# ========================================
print("\n" + "="*60)
print("STEP 4: CALCULATING DATE RANGE INFORMATION")
print("="*60)

# Get start and end dates
start_date = df.index.min()
end_date = df.index.max()

# Calculate total duration
duration = end_date - start_date
years = duration.days // 365
months = (duration.days % 365) // 30
days = (duration.days % 365) % 30
hours = duration.seconds // 3600

print(f"\nDate Range Information:")
print(f"  Start: {start_date}")
print(f"  End: {end_date}")
print(f"  Total Duration: {years} years, {months} months, {days} days, {hours} hours")
print(f"  Total Days: {duration.days}")
print(f"  Total Records: {len(df):,}")


STEP 4: CALCULATING DATE RANGE INFORMATION

Date Range Information:
  Start: 2015-04-25 09:00:00
  End: 2025-11-24 12:00:00
  Total Duration: 10 years, 7 months, 6 days, 3 hours
  Total Days: 3866
  Total Records: 195,892


In [18]:

# ========================================
# SAVE ARTIFACT 1: q3_wrangled_data.csv
# ========================================
print("\n" + "="*60)
print("SAVING ARTIFACTS")
print("="*60)

# Reset index to save datetime as a column (IMPORTANT!)
print("\nSaving wrangled data...")
df_to_save = df.reset_index()
df_to_save.to_csv('output/q3_wrangled_data.csv', index=False)
print("✓ Saved: output/q3_wrangled_data.csv")
print(f"  Columns saved: {df_to_save.shape[1]}")
print(f"  Rows saved: {df_to_save.shape[0]}")



SAVING ARTIFACTS

Saving wrangled data...
✓ Saved: output/q3_wrangled_data.csv
  Columns saved: 26
  Rows saved: 195892


In [19]:

# ========================================
# SAVE ARTIFACT 2: q3_temporal_features.csv
# ========================================
print("\nSaving temporal features...")

# Select temporal features to save (required + optional)
temporal_cols = ['hour', 'day_of_week', 'month', 'year', 'day_name', 'is_weekend']

# Reset index to include datetime as a column
temporal_df = df[temporal_cols].reset_index()
temporal_df.to_csv('output/q3_temporal_features.csv', index=False)
print("✓ Saved: output/q3_temporal_features.csv")
print(f"  Columns saved: {temporal_df.shape[1]}")
print(f"  First column: {temporal_df.columns[0]} (datetime)")
print(f"  Temporal features: {', '.join(temporal_cols)}")

# Display sample of temporal features
print("\nSample of temporal features saved:")
print(temporal_df.head())


Saving temporal features...
✓ Saved: output/q3_temporal_features.csv
  Columns saved: 7
  First column: Measurement Timestamp (datetime)
  Temporal features: hour, day_of_week, month, year, day_name, is_weekend

Sample of temporal features saved:
  Measurement Timestamp  hour  day_of_week  month  year  day_name  is_weekend
0   2015-04-25 09:00:00     9            5      4  2015  Saturday           1
1   2015-04-30 05:00:00     5            3      4  2015  Thursday           0
2   2015-05-22 15:00:00    15            4      5  2015    Friday           0
3   2015-05-22 16:00:00    16            4      5  2015    Friday           0
4   2015-05-22 17:00:00    17            4      5  2015    Friday           0


In [20]:

# ========================================
# SAVE ARTIFACT 3: q3_datetime_info.txt
# ========================================
print("\nSaving datetime information...")

with open('output/q3_datetime_info.txt', 'w') as f:
    f.write("Date Range After Datetime Parsing:\n")
    f.write(f"Start: {start_date}\n")
    f.write(f"End: {end_date}\n")
    f.write(f"Total Duration: {years} years, {months} months, {days} days, {hours} hours\n")
    f.write(f"\nAdditional Information:\n")
    f.write(f"Total Days: {duration.days}\n")
    f.write(f"Total Records: {len(df):,}\n")
    f.write(f"Datetime Column: {datetime_col}\n")

print("✓ Saved: output/q3_datetime_info.txt")


Saving datetime information...
✓ Saved: output/q3_datetime_info.txt
